# Assignment 3: End-to-End Portal Design
## Rangg Roop Order Portal

**Course:** PAI  |  **Team:** [25L-XXXX] and [25L-XXXX]  |  **Date:** November 2026

---

### What this is
A console-based order portal for **Rangg Roop**, a small women's clothing business in Pakistan. Staff log in, record customer orders, search them, update delivery status, and pull a sales report. Everything is saved to CSV files, so the data is still there the next time the portal starts.

### How this notebook is organised
| Section | Content |
|---|---|
| 1 | Setup and configuration |
| 2 | File handling (`load_orders`, `save_orders`, `append_order`) |
| 3 | Users and login |
| 4 | Validation, `add_record`, `search_record` |
| 5 | Update, delete, `generate_report` |
| 6 | Interactive menu (`main`) |
| 7 | **Sample test runs** with visible outputs |
| 8 | Live portal cell (type your own inputs) |

### Requirements map
| Requirement | Where it is shown |
|---|---|
| Functions with parameters, return values, docstrings | Sections 2 to 6 |
| `while True` main loop and `for` loops | `main()` and `generate_report()` / `display_orders()` |
| File handling (read, write, append) | `load_orders`, `save_orders`, `append_order` |
| `try / except / finally` | `load_orders`, `save_orders`, `add_record` callers, `main` |

### Default accounts (for testing)
`admin / admin123` and `staff / staff123`

## 1. Setup and configuration

Everything the portal needs to know up front lives here: file names, the column layout of an order, and the allowed order statuses.
Keeping these in one place means a change (say, adding a new status) is made once, not hunted down across the code.

**Design decision:** an order's *total* is never stored in the file. It is calculated from `quantity * unit_price` whenever needed, so it can never go stale if a quantity is edited.

In [1]:
import csv
import hashlib
import math
import os
from datetime import datetime

ORDERS_FILE = "orders.csv"
USERS_FILE = "users.csv"

ORDER_FIELDS = ["order_id", "customer_name", "city", "item",
                "quantity", "unit_price", "order_date", "status"]
USER_FIELDS = ["username", "password_hash", "role"]
VALID_STATUSES = ["Pending", "Delivered", "Returned"]

## 2. File handling

Three functions handle the orders file:

| Function | Job | File mode |
|---|---|---|
| `load_orders()` | Read every order at startup | read |
| `append_order()` | Add one new order to the end of the file | append (`"a"`) |
| `save_orders()` | Rewrite the whole file after an edit or delete | write (`"w"`) |

`load_orders()` is defensive on purpose. A missing file is created, a permission problem is reported, and a damaged row is skipped with a warning instead of crashing the whole portal. The `finally` block always prints a load summary, whether the load worked or not.

In [2]:
def save_orders(filename, orders):
    """Write the full list of orders to a CSV file (overwrites the file).

    Args:
        filename (str): path of the CSV file.
        orders (list[dict]): every order currently in memory.

    Returns:
        bool: True if saved, False if the file could not be written.
    """
    try:
        with open(filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=ORDER_FIELDS)
            writer.writeheader()
            writer.writerows(orders)
        return True
    except PermissionError:
        print(f"  [error] No permission to write '{filename}'. Close it in Excel and try again.")
        return False
    except OSError as error:
        print(f"  [error] Could not write '{filename}': {error}")
        return False


def append_order(filename, order):
    """Add a single order to the end of the CSV file.

    Writes the header row first if the file is new or empty.

    Args:
        filename (str): path of the CSV file.
        order (dict): one order, with the keys listed in ORDER_FIELDS.

    Returns:
        bool: True if appended, False if the write failed.
    """
    try:
        needs_header = (not os.path.exists(filename)) or os.path.getsize(filename) == 0
        with open(filename, "a", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=ORDER_FIELDS)
            if needs_header:
                writer.writeheader()
            writer.writerow(order)
        return True
    except PermissionError:
        print(f"  [error] No permission to write '{filename}'. Close it in Excel and try again.")
        return False
    except OSError as error:
        print(f"  [error] Could not write '{filename}': {error}")
        return False


def load_orders(filename):
    """Read all orders from a CSV file into a list of dictionaries.

    - Missing file  -> a new file with just the header row is created.
    - No permission -> a message is printed and an empty list is returned.
    - Damaged row   -> that row is skipped with a warning, the rest still load.

    Args:
        filename (str): path of the CSV file.

    Returns:
        list[dict]: orders, with quantity as int and unit_price as float.
    """
    orders = []
    skipped = 0
    try:
        with open(filename, newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            for row_number, row in enumerate(reader, start=2):  # row 1 is the header
                try:
                    order = {field: (row.get(field) or "").strip() for field in ORDER_FIELDS}
                    order["quantity"] = int(order["quantity"])
                    order["unit_price"] = float(order["unit_price"])
                    if not order["order_id"]:
                        raise ValueError("missing order_id")
                    if order["status"] not in VALID_STATUSES:
                        raise ValueError("unknown status")
                    orders.append(order)
                except ValueError:
                    skipped += 1
                    print(f"  [warning] Skipped row {row_number}: data is damaged.")
    except FileNotFoundError:
        print(f"  [info] '{filename}' not found. Creating a new empty file.")
        save_orders(filename, [])
    except PermissionError:
        print(f"  [error] No permission to read '{filename}'.")
    finally:
        print(f"  [load] {len(orders)} order(s) loaded, {skipped} skipped.")
    return orders

## 3. Users and login

Passwords are **never stored as plain text**. `users.csv` keeps a SHA-256 hash of each password, and at login we hash what the person typed and compare the two hashes.

Two roles exist:
- **admin** can do everything, including deleting orders
- **staff** can add, search, update status and view reports, but cannot delete

`check_credentials()` is the pure checking logic (easy to test), and `user_login()` wraps it in a `for` loop that allows 3 attempts.

In [3]:
def hash_password(password):
    """Turn a password into a SHA-256 hash string.

    Args:
        password (str): the plain-text password.

    Returns:
        str: 64-character hexadecimal hash.
    """
    return hashlib.sha256(password.encode("utf-8")).hexdigest()


def default_users():
    """Return the two starter accounts used when users.csv does not exist yet."""
    return [
        {"username": "admin", "password_hash": hash_password("admin123"), "role": "admin"},
        {"username": "staff", "password_hash": hash_password("staff123"), "role": "staff"},
    ]


def load_users(filename):
    """Read user accounts from a CSV file.

    If the file is missing, it is created with the default accounts.

    Args:
        filename (str): path of the users CSV file.

    Returns:
        list[dict]: accounts with keys username, password_hash, role.
    """
    users = []
    try:
        with open(filename, newline="", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                users.append({field: (row.get(field) or "").strip() for field in USER_FIELDS})
    except FileNotFoundError:
        print(f"  [info] '{filename}' not found. Creating it with default accounts.")
        users = default_users()
        try:
            with open(filename, "w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(file, fieldnames=USER_FIELDS)
                writer.writeheader()
                writer.writerows(users)
        except (PermissionError, OSError) as error:
            print(f"  [warning] Could not save '{filename}': {error}. Using defaults in memory only.")
    except PermissionError:
        print(f"  [error] No permission to read '{filename}'.")
    return users


def check_credentials(users, username, password):
    """Check a username and password against the stored accounts.

    Args:
        users (list[dict]): accounts from load_users().
        username (str): typed username.
        password (str): typed plain-text password.

    Returns:
        str | None: the account's role ("admin" or "staff") if correct, otherwise None.
    """
    typed_hash = hash_password(password)
    for user in users:
        if user["username"] == username.strip() and user["password_hash"] == typed_hash:
            return user["role"]
    return None


def user_login(users, max_attempts=3):
    """Ask for a username and password, allowing a limited number of attempts.

    Args:
        users (list[dict]): accounts from load_users().
        max_attempts (int): how many tries before giving up.

    Returns:
        dict | None: {"username": ..., "role": ...} on success, None if all attempts fail.
    """
    for attempt in range(1, max_attempts + 1):
        username = input("Username: ")
        password = input("Password: ")
        role = check_credentials(users, username, password)
        if role is not None:
            print(f"\nWelcome, {username.strip()} ({role}).")
            return {"username": username.strip(), "role": role}
        remaining = max_attempts - attempt
        print(f"  Wrong username or password. {remaining} attempt(s) left.")
    print("Too many failed attempts. Portal locked.")
    return None

## 4. Validation, adding and searching records

Every piece of user input goes through a small `parse_...` function first. Each one either returns a clean value or raises a `ValueError` with a message a human can understand. That keeps the validation rules in one place, and both the menu and `add_record()` reuse them.

Notable details:
- `parse_price()` rejects `nan` and `inf`. Python happily converts the text `"nan"` to a float, and `nan <= 0` is `False`, so a naive check would let it through.
- `next_order_id()` finds the highest existing number and adds one, so IDs stay unique even after deletions.
- `add_record()` validates **everything before** changing the list, so a bad field can never leave a half-added order behind.

In [4]:
def parse_text(raw, label="Value"):
    """Return trimmed text, or raise ValueError if it is empty."""
    cleaned = str(raw).strip()
    if not cleaned:
        raise ValueError(f"{label} cannot be empty.")
    return cleaned


def parse_quantity(raw):
    """Convert input to a whole number greater than zero, or raise ValueError."""
    try:
        value = int(str(raw).strip())
    except ValueError:
        raise ValueError("Quantity must be a whole number, like 1 or 3.")
    if value <= 0:
        raise ValueError("Quantity must be greater than zero.")
    return value


def parse_price(raw):
    """Convert input to a positive, finite number, or raise ValueError."""
    try:
        value = float(str(raw).strip().replace(",", ""))
    except ValueError:
        raise ValueError("Price must be a number, like 4500 or 1899.50.")
    if not math.isfinite(value) or value <= 0:
        raise ValueError("Price must be a real number greater than zero.")
    return value


def parse_date(raw):
    """Check a date is written as YYYY-MM-DD and return it, or raise ValueError."""
    text = str(raw).strip()
    try:
        datetime.strptime(text, "%Y-%m-%d")
    except ValueError:
        raise ValueError("Date must look like 2026-11-15 (year-month-day).")
    return text


def parse_status(raw):
    """Match input to one of VALID_STATUSES (case-insensitive), or raise ValueError."""
    text = str(raw).strip().lower()
    for status in VALID_STATUSES:
        if text == status.lower():
            return status
    raise ValueError("Status must be one of: " + ", ".join(VALID_STATUSES) + ".")


def order_total(order):
    """Return quantity x unit_price for one order."""
    return order["quantity"] * order["unit_price"]


def next_order_id(orders):
    """Work out the next free order ID, such as 'RR-017'.

    Args:
        orders (list[dict]): current orders.

    Returns:
        str: ID one higher than the largest existing RR-number.
    """
    highest = 0
    for order in orders:
        try:
            number = int(order["order_id"].split("-")[1])
        except (IndexError, ValueError):
            continue  # ignore IDs that don't follow the RR-### pattern
        highest = max(highest, number)
    return f"RR-{highest + 1:03d}"


def add_record(orders, customer_name, city, item, quantity, unit_price,
               order_date=None, status="Pending"):
    """Validate the details and add a new order to the list.

    Nothing is added unless every field is valid.

    Args:
        orders (list[dict]): current orders (modified in place).
        customer_name, city, item (str): text fields.
        quantity (int | str): number of units.
        unit_price (float | str): price per unit in PKR.
        order_date (str | None): YYYY-MM-DD, defaults to today.
        status (str): one of VALID_STATUSES, defaults to "Pending".

    Returns:
        dict: the order that was added.

    Raises:
        ValueError: if any field is invalid.
    """
    if order_date is None:
        order_date = datetime.now().strftime("%Y-%m-%d")

    new_order = {
        "order_id": next_order_id(orders),
        "customer_name": parse_text(customer_name, "Customer name").title(),
        "city": parse_text(city, "City").title(),
        "item": parse_text(item, "Item"),
        "quantity": parse_quantity(quantity),
        "unit_price": parse_price(unit_price),
        "order_date": parse_date(order_date),
        "status": parse_status(status),
    }
    orders.append(new_order)
    return new_order


def search_record(orders, keyword):
    """Find orders where the keyword appears in the ID, customer, city, item or status.

    The search ignores upper/lower case, so 'lahore' finds 'Lahore'.

    Args:
        orders (list[dict]): current orders.
        keyword (str): text to look for.

    Returns:
        list[dict]: matching orders (empty list if none, or if keyword is blank).
    """
    keyword = keyword.strip().lower()
    if not keyword:
        return []
    matches = []
    for order in orders:
        searchable = [order["order_id"], order["customer_name"],
                      order["city"], order["item"], order["status"]]
        for value in searchable:
            if keyword in value.lower():
                matches.append(order)
                break  # one hit is enough, avoid listing the same order twice
    return matches

## 5. Updating, deleting and the analytics report

`generate_report()` is where the portal stops being a plain data-entry tool and starts answering business questions.

**Revenue rule used in the report:** a *Returned* order does not count as revenue. Its value is shown separately as `returned_value`, so you can see how much money came back out. *Delivered* revenue (cash actually earned) is also shown on its own from *Pending* revenue (money still to be confirmed).

**Ties are reported, not hidden.** In the sample data three items sell 6 units each. `top_entries()` returns all tied winners instead of letting `max()` quietly pick whichever came first.

`generate_report()` only calculates and returns a dictionary. A separate `print_report()` handles the display. Splitting calculation from printing means the numbers can be tested without parsing printed text.

In [5]:
def find_order(orders, order_id):
    """Return the order with the given ID, or None if it does not exist."""
    wanted = order_id.strip().upper()
    for order in orders:
        if order["order_id"].upper() == wanted:
            return order
    return None


def update_status(orders, order_id, new_status):
    """Change the status of one order.

    Args:
        orders (list[dict]): current orders (modified in place).
        order_id (str): e.g. 'RR-004'.
        new_status (str): one of VALID_STATUSES.

    Returns:
        bool: True if updated, False if no order has that ID.

    Raises:
        ValueError: if new_status is not a valid status.
    """
    new_status = parse_status(new_status)  # validate first
    order = find_order(orders, order_id)
    if order is None:
        return False
    order["status"] = new_status
    return True


def delete_record(orders, order_id):
    """Remove one order from the list.

    Returns:
        bool: True if deleted, False if no order has that ID.
    """
    order = find_order(orders, order_id)
    if order is None:
        return False
    orders.remove(order)
    return True


def top_entries(totals):
    """Return the key with the highest value. If several tie, return all of them joined by ' / '.

    A plain max() would silently pick the first tied key and hide the tie from the reader.

    Args:
        totals (dict): e.g. {"Lahore": 22100.0, "Karachi": 12400.0}

    Returns:
        str | None: the winning key(s), or None if the dictionary is empty.
    """
    if not totals:
        return None
    highest = max(totals.values())
    winners = [key for key, value in totals.items() if value == highest]
    return " / ".join(winners)


def generate_report(orders):
    """Calculate summary statistics for all orders.

    Args:
        orders (list[dict]): current orders.

    Returns:
        dict: total_orders, total_revenue, delivered_revenue, pending_revenue,
        returned_value, return_rate_pct, avg_order_value, revenue_by_city,
        units_by_item, orders_by_status, top_city, best_selling_item.
    """
    report = {
        "total_orders": len(orders),
        "total_revenue": 0.0,
        "delivered_revenue": 0.0,
        "pending_revenue": 0.0,
        "returned_value": 0.0,
        "return_rate_pct": 0.0,
        "avg_order_value": 0.0,
        "revenue_by_city": {},
        "units_by_item": {},
        "orders_by_status": {status: 0 for status in VALID_STATUSES},
        "top_city": None,
        "best_selling_item": None,
    }

    for order in orders:
        total = order_total(order)
        report["orders_by_status"][order["status"]] += 1

        if order["status"] == "Returned":
            report["returned_value"] += total
            continue  # returned orders are not revenue

        report["total_revenue"] += total
        if order["status"] == "Delivered":
            report["delivered_revenue"] += total
        else:
            report["pending_revenue"] += total

        city = order["city"]
        report["revenue_by_city"][city] = report["revenue_by_city"].get(city, 0.0) + total
        item = order["item"]
        report["units_by_item"][item] = report["units_by_item"].get(item, 0) + order["quantity"]

    kept_orders = report["total_orders"] - report["orders_by_status"]["Returned"]
    if report["total_orders"] > 0:
        report["return_rate_pct"] = report["orders_by_status"]["Returned"] / report["total_orders"] * 100
    if kept_orders > 0:
        report["avg_order_value"] = report["total_revenue"] / kept_orders
    report["top_city"] = top_entries(report["revenue_by_city"])
    report["best_selling_item"] = top_entries(report["units_by_item"])
    return report


def print_report(report):
    """Print the dictionary from generate_report() in a readable layout."""
    print("\n" + "=" * 52)
    print("  RANGG ROOP - SALES REPORT")
    print("=" * 52)
    print(f"  Total orders           : {report['total_orders']}")
    print(f"  Total revenue          : PKR {report['total_revenue']:,.0f}")
    print(f"    - Delivered          : PKR {report['delivered_revenue']:,.0f}")
    print(f"    - Pending            : PKR {report['pending_revenue']:,.0f}")
    print(f"  Returned (not counted) : PKR {report['returned_value']:,.0f}")
    print(f"  Return rate            : {report['return_rate_pct']:.1f}%")
    print(f"  Average order value    : PKR {report['avg_order_value']:,.0f}")
    print("\n  Orders by status:")
    for status, count in report["orders_by_status"].items():
        print(f"    {status:<10} {count}")
    print("\n  Revenue by city (highest first):")
    ranked = sorted(report["revenue_by_city"].items(), key=lambda pair: pair[1], reverse=True)
    for city, revenue in ranked:
        print(f"    {city:<12} PKR {revenue:>10,.0f}")
    print("\n  Units sold by item (highest first):")
    ranked = sorted(report["units_by_item"].items(), key=lambda pair: pair[1], reverse=True)
    for item, units in ranked:
        print(f"    {item:<20} {units}")
    print(f"\n  Top city          : {report['top_city']}")
    print(f"  Best-selling item : {report['best_selling_item']}")
    print("=" * 52)


def display_orders(orders):
    """Print a list of orders as a table, with a count at the bottom."""
    if not orders:
        print("  (no orders to show)")
        return
    print(f"\n  {'ID':<8}{'Customer':<18}{'City':<12}{'Item':<20}{'Qty':>4}{'Total (PKR)':>13}  {'Date':<11}{'Status'}")
    print("  " + "-" * 98)
    for order in orders:
        print(f"  {order['order_id']:<8}{order['customer_name']:<18}{order['city']:<12}"
              f"{order['item']:<20}{order['quantity']:>4}{order_total(order):>13,.0f}  "
              f"{order['order_date']:<11}{order['status']}")
    print(f"\n  {len(orders)} order(s) shown.")

## 6. The interactive menu

`main()` ties everything together:

1. load users and log in (3 attempts)
2. load orders from the file
3. run a `while True` loop that shows the menu until the person chooses **0 (Exit)**
4. a `finally` block always prints a goodbye, even if the program is interrupted with Ctrl+C

`ask_until_valid()` keeps asking one question until the answer passes validation, so a typo in the price doesn't force the person to re-type the customer name.

**Role check:** option 6 (delete) is only shown to admins, and it is checked again inside the loop. Hiding a menu line is not security by itself.

In [6]:
def ask_until_valid(prompt, parser):
    """Keep asking until the parser accepts the answer, then return the clean value."""
    while True:
        raw = input(prompt)
        try:
            return parser(raw)
        except ValueError as error:
            print(f"  [!] {error}")


def show_menu(role):
    """Print the menu. The delete option is shown to admins only."""
    print("\n------ RANGG ROOP ORDER PORTAL ------")
    print("  1. Add new order")
    print("  2. View all orders")
    print("  3. Search orders")
    print("  4. Update order status")
    print("  5. Sales report")
    if role == "admin":
        print("  6. Delete an order (admin)")
    print("  0. Exit")


def prompt_add_order(orders, orders_file):
    """Collect details from the user, add the order, and append it to the file."""
    print("\nNew order (press Ctrl+C to cancel)")
    name = ask_until_valid("  Customer name: ", lambda r: parse_text(r, "Customer name"))
    city = ask_until_valid("  City: ", lambda r: parse_text(r, "City"))
    item = ask_until_valid("  Item: ", lambda r: parse_text(r, "Item"))
    quantity = ask_until_valid("  Quantity: ", parse_quantity)
    price = ask_until_valid("  Unit price (PKR): ", parse_price)
    today = datetime.now().strftime("%Y-%m-%d")
    order_date = ask_until_valid(f"  Date [Enter = {today}]: ", lambda r: parse_date(r or today))
    status = ask_until_valid("  Status [Pending/Delivered/Returned, Enter = Pending]: ",
                             lambda r: parse_status(r or "Pending"))

    order = add_record(orders, name, city, item, quantity, price, order_date, status)
    if append_order(orders_file, order):
        print(f"  Saved. New order ID: {order['order_id']}  (total PKR {order_total(order):,.0f})")
    else:
        print(f"  Order {order['order_id']} is in memory but was NOT saved to the file.")


def prompt_search(orders):
    """Ask for a keyword and show the matching orders."""
    keyword = input("  Search (ID, customer, city, item or status): ")
    results = search_record(orders, keyword)
    if results:
        display_orders(results)
    else:
        print("  No matching orders found.")


def prompt_update_status(orders, orders_file):
    """Ask for an order ID and a new status, then save the file."""
    order_id = input("  Order ID to update (e.g. RR-004): ")
    new_status = ask_until_valid("  New status [Pending/Delivered/Returned]: ", parse_status)
    if update_status(orders, order_id, new_status):
        saved = save_orders(orders_file, orders)
        print("  Status updated and saved." if saved else "  Updated in memory only, file NOT saved.")
    else:
        print(f"  No order found with ID '{order_id.strip()}'.")


def prompt_delete(orders, orders_file):
    """Ask for an order ID, confirm, delete it, then save the file."""
    order_id = input("  Order ID to delete: ")
    order = find_order(orders, order_id)
    if order is None:
        print(f"  No order found with ID '{order_id.strip()}'.")
        return
    confirm = input(f"  Type YES to delete {order['order_id']} ({order['customer_name']}): ")
    if confirm.strip() == "YES":
        delete_record(orders, order['order_id'])
        saved = save_orders(orders_file, orders)
        print("  Order deleted and saved." if saved else "  Deleted in memory only, file NOT saved.")
    else:
        print("  Cancelled. Nothing was deleted.")


def main(orders_file=ORDERS_FILE, users_file=USERS_FILE):
    """Run the portal: login, then a menu loop until the user exits."""
    print("=" * 40)
    print("   RANGG ROOP ORDER PORTAL")
    print("=" * 40)
    try:
        users = load_users(users_file)
        session = user_login(users)
        if session is None:
            return
        orders = load_orders(orders_file)

        while True:
            show_menu(session["role"])
            choice = input("Choose an option: ").strip()

            if choice == "1":
                prompt_add_order(orders, orders_file)
            elif choice == "2":
                display_orders(orders)
            elif choice == "3":
                prompt_search(orders)
            elif choice == "4":
                prompt_update_status(orders, orders_file)
            elif choice == "5":
                print_report(generate_report(orders))
            elif choice == "6" and session["role"] == "admin":
                prompt_delete(orders, orders_file)
            elif choice == "6":
                print("  Only admins can delete orders.")
            elif choice == "0":
                break
            else:
                print("  Invalid choice. Please type a number from the menu.")
    except (KeyboardInterrupt, EOFError):
        print("\n  Input interrupted. Closing the portal.")
    finally:
        print("Session closed. Goodbye.")

## 7. Sample test runs

These cells prove each requirement works, using **pre-written inputs** so the output is visible without anyone typing. All tests run on a **copy** of the data (`orders_test.csv`), so the real `orders.csv` stays untouched however many times this notebook is re-run.

In [7]:
import shutil
from unittest.mock import patch

TEST_FILE = "orders_test.csv"
shutil.copy(ORDERS_FILE, TEST_FILE)  # fresh copy every run
print(f"Test copy created: {TEST_FILE}")

Test copy created: orders_test.csv


### Test 1: Startup loads existing records from the file

In [8]:
test_orders = load_orders(TEST_FILE)
display_orders(test_orders[:5])  # first five rows only

  [load] 16 order(s) loaded, 0 skipped.

  ID      Customer          City        Item                 Qty  Total (PKR)  Date       Status
  --------------------------------------------------------------------------------------------------
  RR-001  Ayesha Khan       Lahore      Lawn 3-Piece           2        9,000  2026-10-20 Delivered
  RR-002  Hira Malik        Karachi     Embroidered Kurta      1        3,800  2026-10-22 Delivered
  RR-003  Sana Tariq        Islamabad   Khaddar Shawl          1        3,200  2026-10-24 Delivered
  RR-004  Maryam Iqbal      Lahore      Chiffon Dupatta        3        5,400  2026-10-27 Delivered
  RR-005  Fatima Noor       Gujranwala  Lawn 3-Piece           1        4,500  2026-10-29 Returned

  5 order(s) shown.


### Test 2: Login (correct and wrong credentials)

In [9]:
users = load_users(USERS_FILE)
print("admin  / admin123 ->", check_credentials(users, "admin", "admin123"))
print("staff  / staff123 ->", check_credentials(users, "staff", "staff123"))
print("admin  / wrong    ->", check_credentials(users, "admin", "wrong"))
print("nobody / admin123 ->", check_credentials(users, "nobody", "admin123"))

admin  / admin123 -> admin
staff  / staff123 -> staff
admin  / wrong    -> None
nobody / admin123 -> None


### Test 3: `add_record()` with valid data, then `append_order()` writes it to the file
The file grows by one line without being rewritten.

In [10]:
lines_before = len(open(TEST_FILE, encoding="utf-8").read().splitlines())
new_order = add_record(test_orders, "sara ahmed", "lahore", "Lawn 3-Piece", 2, 4500, "2026-11-15", "Pending")
append_order(TEST_FILE, new_order)
lines_after = len(open(TEST_FILE, encoding="utf-8").read().splitlines())

print("Added:", new_order)
print(f"File lines before: {lines_before}  |  after: {lines_after}")

Added: {'order_id': 'RR-017', 'customer_name': 'Sara Ahmed', 'city': 'Lahore', 'item': 'Lawn 3-Piece', 'quantity': 2, 'unit_price': 4500.0, 'order_date': '2026-11-15', 'status': 'Pending'}
File lines before: 17  |  after: 18


### Test 4: Invalid input is rejected with a clear message (no crash)
Each bad value below would normally cause a crash or silently corrupt the file. Here every one is caught, and the order list stays unchanged.

In [11]:
bad_inputs = [
    ("quantity = 'abc'",       dict(quantity="abc",  unit_price=4500,   order_date="2026-11-15", status="Pending")),
    ("quantity = -2",          dict(quantity=-2,     unit_price=4500,   order_date="2026-11-15", status="Pending")),
    ("price = 'free'",         dict(quantity=1,      unit_price="free", order_date="2026-11-15", status="Pending")),
    ("price = 'nan'",          dict(quantity=1,      unit_price="nan",  order_date="2026-11-15", status="Pending")),
    ("price = 0",              dict(quantity=1,      unit_price=0,      order_date="2026-11-15", status="Pending")),
    ("date = '15/11/2026'",    dict(quantity=1,      unit_price=4500,   order_date="15/11/2026", status="Pending")),
    ("date = '2026-02-30'",    dict(quantity=1,      unit_price=4500,   order_date="2026-02-30", status="Pending")),
    ("status = 'Shipped'",     dict(quantity=1,      unit_price=4500,   order_date="2026-11-15", status="Shipped")),
]

count_before = len(test_orders)
for label, fields in bad_inputs:
    try:
        add_record(test_orders, "Test Customer", "Lahore", "Cotton Trouser", **fields)
        print(f"{label:<22} -> ACCEPTED (this would be a bug)")
    except ValueError as error:
        print(f"{label:<22} -> rejected: {error}")

print(f"\nOrders before: {count_before}  |  after: {len(test_orders)}  (should be equal)")

quantity = 'abc'       -> rejected: Quantity must be a whole number, like 1 or 3.
quantity = -2          -> rejected: Quantity must be greater than zero.
price = 'free'         -> rejected: Price must be a number, like 4500 or 1899.50.
price = 'nan'          -> rejected: Price must be a real number greater than zero.
price = 0              -> rejected: Price must be a real number greater than zero.
date = '15/11/2026'    -> rejected: Date must look like 2026-11-15 (year-month-day).
date = '2026-02-30'    -> rejected: Date must look like 2026-11-15 (year-month-day).
status = 'Shipped'     -> rejected: Status must be one of: Pending, Delivered, Returned.

Orders before: 17  |  after: 17  (should be equal)


### Test 5: `search_record()`
Case-insensitive, searches ID, customer, city, item and status.

In [12]:
for keyword in ["lahore", "KHADDAR", "RR-010", "zzz-no-match"]:
    results = search_record(test_orders, keyword)
    print(f"Search '{keyword}': {len(results)} match(es)")
display_orders(search_record(test_orders, "lahore"))

Search 'lahore': 5 match(es)
Search 'KHADDAR': 3 match(es)
Search 'RR-010': 1 match(es)
Search 'zzz-no-match': 0 match(es)

  ID      Customer          City        Item                 Qty  Total (PKR)  Date       Status
  --------------------------------------------------------------------------------------------------
  RR-001  Ayesha Khan       Lahore      Lawn 3-Piece           2        9,000  2026-10-20 Delivered
  RR-004  Maryam Iqbal      Lahore      Chiffon Dupatta        3        5,400  2026-10-27 Delivered
  RR-008  Ayesha Khan       Lahore      Khaddar Shawl          1        3,200  2026-11-03 Delivered
  RR-013  Mehwish Butt      Lahore      Lawn 3-Piece           1        4,500  2026-11-09 Pending
  RR-017  Sara Ahmed        Lahore      Lawn 3-Piece           2        9,000  2026-11-15 Pending

  5 order(s) shown.


### Test 6: Update status and delete, then `save_orders()` rewrites the file

In [13]:
print("Update RR-009 to Delivered :", update_status(test_orders, "RR-009", "Delivered"))
print("Update RR-999 (missing)    :", update_status(test_orders, "RR-999", "Delivered"))
try:
    update_status(test_orders, "RR-009", "Lost in post")
except ValueError as error:
    print("Update with bad status     : rejected ->", error)

print("Delete RR-005              :", delete_record(test_orders, "RR-005"))
print("Delete RR-999 (missing)    :", delete_record(test_orders, "RR-999"))
print("Saved to file              :", save_orders(TEST_FILE, test_orders))

Update RR-009 to Delivered : True
Update RR-999 (missing)    : False
Update with bad status     : rejected -> Status must be one of: Pending, Delivered, Returned.
Delete RR-005              : True
Delete RR-999 (missing)    : False
Saved to file              : True


### Test 7: `generate_report()` and a cross-check
The report is printed first. Then the same headline numbers are recomputed in a **completely different way** (a list comprehension) to confirm they agree. Checking a result two ways is a habit worth keeping in analytics work.

In [14]:
report = generate_report(test_orders)
print_report(report)


  RANGG ROOP - SALES REPORT
  Total orders           : 16
  Total revenue          : PKR 83,800
    - Delivered          : PKR 47,700
    - Pending            : PKR 36,100
  Returned (not counted) : PKR 7,600
  Return rate            : 6.2%
  Average order value    : PKR 5,587

  Orders by status:
    Pending    6
    Delivered  9
    Returned   1

  Revenue by city (highest first):
    Lahore       PKR     31,100
    Rawalpindi   PKR     13,500
    Karachi      PKR     12,400
    Faisalabad   PKR      9,600
    Islamabad    PKR      6,800
    Gujranwala   PKR      6,000
    Multan       PKR      4,400

  Units sold by item (highest first):
    Lawn 3-Piece         8
    Cotton Trouser       6
    Printed Kurti        6
    Chiffon Dupatta      5
    Khaddar Shawl        4
    Embroidered Kurta    1

  Top city          : Lahore
  Best-selling item : Lawn 3-Piece


In [15]:
check_revenue = sum(o["quantity"] * o["unit_price"] for o in test_orders if o["status"] != "Returned")
check_returned = sum(o["quantity"] * o["unit_price"] for o in test_orders if o["status"] == "Returned")

print(f"Revenue  : report = {report['total_revenue']:,.0f}  |  recomputed = {check_revenue:,.0f}  |  match = {report['total_revenue'] == check_revenue}")
print(f"Returned : report = {report['returned_value']:,.0f}  |  recomputed = {check_returned:,.0f}  |  match = {report['returned_value'] == check_returned}")
print(f"Orders   : report = {report['total_orders']}  |  list length = {len(test_orders)}  |  match = {report['total_orders'] == len(test_orders)}")

Revenue  : report = 83,800  |  recomputed = 83,800  |  match = True
Returned : report = 7,600  |  recomputed = 7,600  |  match = True
Orders   : report = 16  |  list length = 16  |  match = True


### Test 7b: Ties are reported, not hidden
The real seed data (`orders.csv`) has a three-way tie for best-selling item. A plain `max()` would show only one of them.

In [16]:
seed_report = generate_report(load_orders(ORDERS_FILE))
print("Units by item   :", seed_report["units_by_item"])
print("Best-selling    :", seed_report["best_selling_item"])
print("Plain max() would have said:", max(seed_report["units_by_item"], key=seed_report["units_by_item"].get), "(hides the tie)")
print("Empty data      :", top_entries({}))

  [load] 16 order(s) loaded, 0 skipped.
Units by item   : {'Lawn 3-Piece': 6, 'Embroidered Kurta': 1, 'Khaddar Shawl': 4, 'Chiffon Dupatta': 5, 'Cotton Trouser': 6, 'Printed Kurti': 6}
Best-selling    : Lawn 3-Piece / Cotton Trouser / Printed Kurti
Plain max() would have said: Lawn 3-Piece (hides the tie)
Empty data      : None


### Test 8: File error handling
Three situations that would crash an unprotected program.

In [17]:
# 8a. Missing file: it should be created automatically, not crash.
import os
MISSING = "does_not_exist_demo.csv"
if os.path.exists(MISSING):
    os.remove(MISSING)
print("8a. Loading a file that does not exist:")
result = load_orders(MISSING)
print("    Returned:", result, "| file now exists:", os.path.exists(MISSING))
os.remove(MISSING)

8a. Loading a file that does not exist:
  [info] 'does_not_exist_demo.csv' not found. Creating a new empty file.
  [load] 0 order(s) loaded, 0 skipped.
    Returned: [] | file now exists: True


In [18]:
# 8b. Damaged rows: the bad ones are skipped, the good ones still load.
DAMAGED = "damaged_demo.csv"
with open(DAMAGED, "w", encoding="utf-8") as f:
    f.write("order_id,customer_name,city,item,quantity,unit_price,order_date,status\n")
    f.write("RR-001,Good Customer,Lahore,Lawn 3-Piece,2,4500,2026-11-01,Pending\n")
    f.write("RR-002,Bad Quantity,Lahore,Lawn 3-Piece,two,4500,2026-11-01,Pending\n")
    f.write("RR-003,Bad Price,Lahore,Lawn 3-Piece,1,abc,2026-11-01,Pending\n")
    f.write("RR-004,Bad Status,Lahore,Lawn 3-Piece,1,4500,2026-11-01,Shipped\n")
    f.write("RR-005,Short Row\n")
print("8b. Loading a file with damaged rows:")
loaded = load_orders(DAMAGED)
print("    Rows that survived:", [o["order_id"] for o in loaded])
os.remove(DAMAGED)

8b. Loading a file with damaged rows:
  [warning] Skipped row 3: data is damaged.
  [warning] Skipped row 4: data is damaged.
  [warning] Skipped row 5: data is damaged.
  [warning] Skipped row 6: data is damaged.
  [load] 1 order(s) loaded, 4 skipped.
    Rows that survived: ['RR-001']


In [19]:
# 8c. PermissionError: simulated, because a real one depends on the operating system
# (on Windows it happens when the CSV is open in Excel).
# patch.dict() temporarily swaps in an open() that always refuses, just for this block.
def locked_open(*args, **kwargs):
    raise PermissionError("file is locked")

print("8c. Saving when the file is locked (simulated):")
with patch.dict(globals(), {"open": locked_open}):
    print("    save_orders returned  :", save_orders(TEST_FILE, test_orders))
    print("    append_order returned :", append_order(TEST_FILE, test_orders[0]))
print("    (False = failed safely with a message, no crash)")

8c. Saving when the file is locked (simulated):
  [error] No permission to write 'orders_test.csv'. Close it in Excel and try again.
    save_orders returned  : False
  [error] No permission to write 'orders_test.csv'. Close it in Excel and try again.
    append_order returned : False
    (False = failed safely with a message, no crash)


### Test 9: A complete portal session through `main()`
This drives the real `while True` menu with a scripted list of answers. The helper below prints each prompt next to the answer, so the output reads like a real terminal session. It includes a wrong password, a bad price, a staff-only restriction, and a clean exit.

In [20]:
def scripted_input(answers):
    """Return a fake input() that feeds answers one by one and echoes them like a terminal."""
    feed = iter(answers)
    def fake_input(prompt=""):
        answer = next(feed)
        print(f"{prompt}{answer}")
        return answer
    return fake_input

session_answers = [
    "staff", "wrongpass",            # failed login attempt
    "staff", "staff123",             # successful login as staff
    "1",                             # add an order
    "Mahnoor Zia", "Gujranwala", "Printed Kurti",
    "two", "2",                      # bad quantity, then fixed
    "2200",
    "", "",                          # accept today's date and Pending status
    "3", "gujranwala",               # search
    "6",                             # staff tries to delete -> blocked (no ID prompt appears)
    "7",                             # invalid menu choice
    "0",                             # exit
]

shutil.copy(ORDERS_FILE, TEST_FILE)  # reset test data before the session
with patch("builtins.input", scripted_input(session_answers)):
    main(TEST_FILE, USERS_FILE)

   RANGG ROOP ORDER PORTAL
Username: staff
Password: wrongpass
  Wrong username or password. 2 attempt(s) left.
Username: staff
Password: staff123

Welcome, staff (staff).
  [load] 16 order(s) loaded, 0 skipped.

------ RANGG ROOP ORDER PORTAL ------
  1. Add new order
  2. View all orders
  3. Search orders
  4. Update order status
  5. Sales report
  0. Exit
Choose an option: 1

New order (press Ctrl+C to cancel)
  Customer name: Mahnoor Zia
  City: Gujranwala
  Item: Printed Kurti
  Quantity: two
  [!] Quantity must be a whole number, like 1 or 3.
  Quantity: 2
  Unit price (PKR): 2200
  Date [Enter = 2026-11-15]: 
  Status [Pending/Delivered/Returned, Enter = Pending]: 
  Saved. New order ID: RR-017  (total PKR 4,400)

------ RANGG ROOP ORDER PORTAL ------
  1. Add new order
  2. View all orders
  3. Search orders
  4. Update order status
  5. Sales report
  0. Exit
Choose an option: 3
  Search (ID, customer, city, item or status): gujranwala

  ID      Customer          City      

### Test 10: The data file after all of that
Proof that the session's new order was written to disk.

In [21]:
with open(TEST_FILE, encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if line_number == 1 or line_number > 15:  # header plus the last few rows
            print(line.rstrip())

order_id,customer_name,city,item,quantity,unit_price,order_date,status
RR-015,Hira Malik,Karachi,Khaddar Shawl,2,3200.0,2026-11-11,Pending
RR-016,Amna Waheed,Faisalabad,Printed Kurti,3,2200.0,2026-11-12,Pending
RR-017,Mahnoor Zia,Gujranwala,Printed Kurti,2,2200.0,2026-11-15,Pending


### Test summary

| # | What was tested | Result |
|---|---|---|
| 1 | Startup load from CSV | Pass |
| 2 | Login with right and wrong credentials | Pass |
| 3 | Add valid order, append to file | Pass |
| 4 | 8 kinds of invalid input rejected | Pass |
| 5 | Case-insensitive search | Pass |
| 6 | Update, delete, rewrite file | Pass |
| 7 | Report numbers cross-checked a second way, ties reported | Pass |
| 8 | Missing file, damaged rows, permission error | Pass |
| 9 | Full scripted menu session | Pass |
| 10 | Data persisted in file | Pass |

## 8. Run the live portal

Set `RUN_LIVE = True` and run the cell to use the portal yourself. Type your answers in the box that appears under the cell. This uses the **real** `orders.csv`, so anything you add is saved permanently.

It is switched off by default so that "Run All" never stops and waits for typing.

To use it outside Jupyter, run `python portal.py` in a terminal. The behaviour is identical.

In [22]:
RUN_LIVE = False

if RUN_LIVE:
    main()
else:
    print("Live portal is switched off. Set RUN_LIVE = True and re-run this cell to use it.")

Live portal is switched off. Set RUN_LIVE = True and re-run this cell to use it.
